### **Sobre las RNN y sus propiedades**

Este cuaderno trata los siguientes temas:

1. **Compartición de pesos en una red base**  
2. **Preparación del conjunto de nombres por idioma**  
3. **Embeddings y primera RNN**  
4. **Padding y packed sequences**  
5. **RNN más profundas y bidireccionales**  
6. **Puente a la Semana 5: atención sobre secuencias**

La idea pedagógica es pasar de una representación secuencial básica a una explicación de por qué las RNN tienen límites en secuencias largas y por qué eso motiva el uso de atención.



#### **1. Compartición de pesos en una red base**

Este bloque usa **MNIST** como ejemplo sencillo para introducir una idea importante:  
**reutilizar parámetros** en distintas partes de una red.

Aunque aquí todavía no trabajamos con lenguaje, esta intuición es útil porque en NLP luego veremos arquitecturas que reutilizan mecanismos sobre todos los tokens de una secuencia.

In [ ]:
# Importaciones principales para trabajar con PyTorch, visualización y métricas.
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision 
from torchvision import transforms

from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.pyplot import imshow

import pandas as pd

from sklearn.metrics import accuracy_score

import time

from utils import train_simple_network, Flatten

In [ ]:
# Configuración para mostrar los gráficos dentro del notebook.
%matplotlib inline

try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('png', 'pdf')
except Exception:
    pass

In [ ]:
# Fijamos semilla y opciones deterministas para facilitar la reproducibilidad.
torch.backends.cudnn.deterministic = True
from utils import set_seed
set_seed(42)

In [ ]:
# Detectamos automáticamente si hay una GPU disponible.
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [ ]:
# Preparamos un ejemplo base con MNIST para introducir la idea de compartir pesos.
mnist_data_train = torchvision.datasets.MNIST("./data", train=True, download=True, transform=transforms.ToTensor())
mnist_data_test = torchvision.datasets.MNIST("./data", train=False, download=True, transform=transforms.ToTensor())
    
mnist_train_loader = DataLoader(mnist_data_train, batch_size=64, shuffle=True)
mnist_test_loader = DataLoader(mnist_data_test, batch_size=64)

# Número de valores de entrada
D = 28*28 #28 * 28 images 
# Tamaño de la capa oculta
n = 256 
# Número de canales de entrada
C = 1
# Número de clases
classes = 10

# Creamos el modelo base 
model_regular = nn.Sequential(
  Flatten(), 
  nn.Linear(D, n), 
  nn.Tanh(),
  nn.Linear(n, n), 
  nn.Tanh(),
  nn.Linear(n, n), 
  nn.Tanh(),
  nn.Linear(n, classes),
)

In [ ]:
# Entrenamos el modelo base y registramos su desempeño.
loss_func = nn.CrossEntropyLoss()
regular_results = train_simple_network(model_regular, loss_func, mnist_train_loader, test_loader=mnist_test_loader, score_funcs={'Accuracy': accuracy_score}, device=device, epochs=10, disable_tqdm=True)

In [ ]:
# Definimos una capa reutilizada para mostrar cómo se pueden compartir parámetros.
# Creamos la capa de pesos que reutilizaremos para compartir parámetros. 
h_2 = nn.Linear(n, n)
model_shared = nn.Sequential(
  Flatten(), 
  nn.Linear(D, n), 
  nn.Tanh(),
  h_2, # Primer uso
  nn.Tanh(),
  h_2, # Segundo uso: ahora compartimos los mismos pesos
  nn.Tanh(),
  nn.Linear(n, classes),
)

In [ ]:
# Entrenamos el modelo con pesos compartidos.
shared_results = train_simple_network(model_shared, loss_func, mnist_train_loader, test_loader=mnist_test_loader, score_funcs={'Accuracy': accuracy_score}, device=device, epochs=10, disable_tqdm=True)

In [ ]:
# Comparamos el desempeño del modelo normal frente al modelo con pesos compartidos.
# Ahora graficamos los resultados y los comparamos
sns.lineplot(x='epoch', y='test Accuracy', data=regular_results, label='Normal')
sns.lineplot(x='epoch', y='test Accuracy', data=shared_results, label='Compartida')

#### **2. Preparación de datos secuenciales a nivel de caracteres**
Ahora sí pasamos a un problema más cercano a NLP: **clasificar nombres por idioma**.  
Para eso:

- limpiamos caracteres,
- construimos un vocabulario,
- transformamos cada nombre en una secuencia de índices,
- y preparamos un `Dataset` para PyTorch.

Este paso muestra cómo convertir texto en una forma que una red neuronal pueda procesar.

In [ ]:
# Descargamos un conjunto de nombres por idioma para pasar a una tarea secuencial real.
zip_file_url = "https://download.pytorch.org/tutorial/data.zip"

import requests, zipfile, io
r = requests.get(zip_file_url, timeout=30)
r.raise_for_status()
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall()

# El archivo zip está organizado como data/names/[LANG].txt, donde [LANG] representa un idioma.


In [ ]:
# Limpiamos texto, construimos el alfabeto y cargamos los nombres por idioma.
name_language_data = {}

# Eliminamos caracteres Unicode especiales para simplificar el procesamiento.
# Por ejemplo, convertir algo como "Ślusàrski" en "Slusarski".
import unicodedata
import string

all_letters = string.ascii_letters + " .,;'"
n_letters = len(all_letters)
alphabet = {}
for i in range(n_letters):
    alphabet[all_letters[i]] = i
    
# Convierte una cadena Unicode a ASCII plano.
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in all_letters
    )

# Recorremos cada idioma, abrimos el archivo correspondiente y leemos todas sus líneas. 
for zip_path in z.namelist():
    if "data/names/" in zip_path and zip_path.endswith(".txt"):
        lang = zip_path[len("data/names/"):-len(".txt")]
        with z.open(zip_path) as myfile:
            lang_names = [unicodeToAscii(line).lower() for line in str(myfile.read(), encoding='utf-8').strip().split("\n")]
            name_language_data[lang] = lang_names
        print(lang, ": ", len(lang_names))  # Mostramos también el nombre del idioma.

In [ ]:
# Definimos un Dataset a nivel de caracteres para clasificación de nombres por idioma.
class LanguageNameDataset(Dataset):
    
    def __init__(self, lang_name_dict, vocabulary):
        self.label_names = [x for x in lang_name_dict.keys()]
        self.data = []
        self.labels = []
        self.vocabulary = vocabulary
        for y, language in enumerate(self.label_names):
            for sample in lang_name_dict[language]:
                self.data.append(sample)
                self.labels.append(y)
        
    def __len__(self):
        return len(self.data)
    
    def string2InputVec(self, input_string):
        """
        Este método convierte cualquier cadena de entrada en un vector de enteros largos según el vocabulario del objeto. 
        input_string: cadena que se convertirá en un tensor
        """
        T = len(input_string)  # Longitud de la cadena en caracteres.
        
        # Creamos un nuevo tensor para guardar el resultado.
        name_vec = torch.zeros((T), dtype=torch.long)
        # Recorremos la cadena y colocamos el índice correspondiente en el tensor.
        for pos, character in enumerate(input_string):
            name_vec[pos] = self.vocabulary[character]
            
        return name_vec
    
    def __getitem__(self, idx):
        name = self.data[idx]
        label = self.labels[idx]

        return self.string2InputVec(name), label

In [ ]:
# Construimos el dataset y lo separamos en entrenamiento y prueba.
dataset = LanguageNameDataset(name_language_data, alphabet)

split_generator = torch.Generator().manual_seed(42)
train_data, test_data = torch.utils.data.random_split(
    dataset,
    (len(dataset) - 300, 300),
    generator=split_generator,
)
train_loader = DataLoader(train_data, batch_size=1, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1, shuffle=False)


#### **3. Embeddings y primera RNN**

En esta sección el notebook muestra dos ideas clave:

- un **embedding** convierte índices discretos en vectores densos;
- una **RNN** procesa la secuencia paso a paso y resume la información en estados ocultos.


In [ ]:
# Ejemplo rápido: una capa de embeddings convierte índices discretos en vectores densos.
with torch.no_grad():
    input_sequence = torch.tensor([0, 1, 1, 0, 2], dtype=torch.long)
    embd = nn.Embedding(3, 2)
    x_seq = embd(input_sequence)
    print(input_sequence.shape, x_seq.shape)
    print(x_seq)

In [ ]:
# Módulo auxiliar para extraer el último estado oculto de una RNN o LSTM.
class LastTimeStep(nn.Module):
    """
    Clase para extraer las activaciones ocultas del último paso temporal tras
    la salida de un módulo RNN de PyTorch. 
    """
    def __init__(self, rnn_layers=1, bidirectional=False):
        super(LastTimeStep, self).__init__()
        self.rnn_layers = rnn_layers
        if bidirectional:
            self.num_driections = 2
        else:
            self.num_driections = 1    
    
    def forward(self, input):
        # El resultado es una tupla de la forma (out, h_t)
        # o una tupla de la forma (out, (h_t, c_t))
        rnn_output = input[0]
        last_step = input[1] #this will be h_t
        if(type(last_step) == tuple):  # Si viene de una LSTM, recibimos también c_t.
            last_step = last_step[0]  # En ese caso, h_t es el primer elemento de la tupla.
        batch_size = last_step.shape[1]  # Según la documentación: (num_layers * num_directions, batch, hidden_size)
        # Reorganizamos dimensiones para separar capas y direcciones. 
        last_step = last_step.view(self.rnn_layers, self.num_driections, batch_size, -1)
        # Nos quedamos con los resultados de la última capa
        last_step = last_step[self.rnn_layers-1] 
        # Reordenamos para que el batch quede primero
        last_step = last_step.permute(1, 0, 2)
        # Finalmente, aplanamos las últimas dos dimensiones en una sola
        return last_step.reshape(batch_size, -1)

In [ ]:
# Definimos una RNN simple para clasificar nombres a partir de secuencias de caracteres.
D = 64
vocab_size = len(alphabet)
hidden_nodes = 256
classes = len(dataset.label_names)

first_rnn = nn.Sequential(
  nn.Embedding(vocab_size, D),  # (B, T) -> (B, T, D)
  nn.RNN(D, hidden_nodes, batch_first=True),  # (B, T, D) -> ((B, T, D), (S, B, D))
  # La activación tanh ya está incorporada en la RNN, así que no hace falta añadirla aparte.
  LastTimeStep(),  # Tomamos la salida final de la secuencia y la reducimos a un vector por ejemplo, (B, D)
  nn.Linear(hidden_nodes, classes),  # (B, D) -> (B, classes)
)

In [ ]:
# Entrenamos la primera RNN usando batch size igual a 1.
loss_func = nn.CrossEntropyLoss()
batch_one_train = train_simple_network(first_rnn, loss_func, train_loader, test_loader=test_loader, score_funcs={'Accuracy': accuracy_score}, device=device, epochs=5, disable_tqdm=True)

In [ ]:
# Visualizamos la precisión de la RNN simple.
sns.lineplot(x='epoch', y='test Accuracy', data=batch_one_train, label='RNN')

In [ ]:
# Probamos el modelo con un nombre de ejemplo y observamos la distribución de probabilidades.
pred_rnn = first_rnn.to("cpu").eval()
with torch.no_grad():
    preds = F.softmax(pred_rnn(dataset.string2InputVec("frank").reshape(1,-1)), dim=-1)
    for class_id in range(len(dataset.label_names)):
        print(dataset.label_names[class_id], ":", preds[0,class_id].item()*100 , "%")

#### **4. Secuencias de longitud variable: padding y packed sequences**

En NLP, casi nunca todas las secuencias tienen la misma longitud.  
Por eso se introduce:

- **padding**, para igualar tamaños dentro de un batch;
- **packed sequences**, para evitar cómputo inútil sobre posiciones rellenas.

Este bloque es importante porque mejora la eficiencia del entrenamiento y prepara una implementación más realista.

In [ ]:
# Función para padding y packed sequences: permite agrupar secuencias de distinta longitud.
def pad_and_pack(batch):
    # 1, 2 y 3: organizamos longitudes, entradas y etiquetas en listas separadas.
    input_tensors = []
    labels = []
    lengths = []
    for x, y in batch:
        input_tensors.append(x)
        labels.append(y)
        lengths.append(x.shape[0])  # Suponemos que la forma es (T, *).
    # 4: creamos la versión con padding de la entrada.
    x_padded = torch.nn.utils.rnn.pad_sequence(input_tensors, batch_first=False)
    # 5: creamos la versión empaquetada a partir del padding y las longitudes.
    x_packed = torch.nn.utils.rnn.pack_padded_sequence(x_padded, lengths, batch_first=False, enforce_sorted=False)
    # Convertimos las etiquetas en un tensor.
    y_batched = torch.as_tensor(labels, dtype=torch.long)
    # 6: devolvemos una tupla con la entrada empaquetada y sus etiquetas.
    return x_packed, y_batched


In [ ]:
# Adaptador para aplicar embeddings también cuando la entrada es una PackedSequence.
class EmbeddingPackable(nn.Module):
    """
    La capa Embedding de PyTorch no admite directamente objetos PackedSequence.
    Esta clase envoltorio resuelve ese problema. Si la entrada es normal,
    usa la capa Embedding habitual. De lo contrario, opera sobre la secuencia
    empaquetada para devolver una nueva PackedSequence con el resultado correcto. 
    """
    def __init__(self, embd_layer):
        super(EmbeddingPackable, self).__init__()
        self.embd_layer = embd_layer 
    
    def forward(self, input):
        if type(input) == torch.nn.utils.rnn.PackedSequence:
            # Primero desempaquetamos la entrada, 
            sequences, lengths = torch.nn.utils.rnn.pad_packed_sequence(input.cpu(), batch_first=True)
            # Luego aplicamos embedding
            sequences = self.embd_layer(sequences.to(input.data.device))
            # Y la volvemos a empaquetar
            return torch.nn.utils.rnn.pack_padded_sequence(sequences, lengths.cpu(), 
                                                           batch_first=True, enforce_sorted=False)
        else:  # Aplicamos embedding sobre datos normales
            return self.embd_layer(input)


In [ ]:
# Creamos DataLoaders por lotes usando la función de colación personalizada.
B = 16
train_packed_loader = DataLoader(train_data, batch_size=B, shuffle=True, collate_fn=pad_and_pack)
test_packed_loader = DataLoader(test_data, batch_size=B, shuffle=False, collate_fn=pad_and_pack)


In [ ]:
# Definimos una RNN que trabaja con secuencias empaquetadas para ganar eficiencia.
rnn_packed = nn.Sequential(
  EmbeddingPackable(nn.Embedding(vocab_size, D)), #(B, T) -> (B, T, D)
  nn.RNN(D, hidden_nodes, batch_first=True),  # (B, T, D) -> ((B, T, D), (S, B, D))
  LastTimeStep(),  # Tomamos la salida final de la secuencia y la reducimos a un vector por ejemplo, (B, D)
  nn.Linear(hidden_nodes, classes),  # (B, D) -> (B, classes)
)

rnn_packed.to(device)

In [ ]:
# Entrenamos la versión empaquetada de la RNN.
packed_train = train_simple_network(rnn_packed, loss_func, train_packed_loader, test_loader=test_packed_loader, score_funcs={'Accuracy': accuracy_score}, device=device, epochs=20, disable_tqdm=True)

In [ ]:
# Comparamos la precisión entre la RNN con batch=1 y la RNN con entradas empaquetadas.
sns.lineplot(x='epoch', y='test Accuracy', data=batch_one_train, label='RNN: Batch=1')
sns.lineplot(x='epoch', y='test Accuracy', data=packed_train, label='RNN: Packed Input')

In [ ]:
# Comparamos tiempo total y precisión para ver la mejora de eficiencia.
sns.lineplot(x='total time', y='test Accuracy', data=batch_one_train, label='RNN: Batch=1')
sns.lineplot(x='total time', y='test Accuracy', data=packed_train, label='RNN: Packed Input')

In [ ]:
# Probamos la RNN empaquetada con un nombre de ejemplo.
pred_rnn = rnn_packed.to("cpu").eval()

with torch.no_grad():
    preds = F.softmax(pred_rnn(dataset.string2InputVec("frank").reshape(1,-1)), dim=-1)
    for class_id in range(len(dataset.label_names)):
        print(dataset.label_names[class_id], ":", preds[0,class_id].item()*100 , "%")

#### **5. Variantes de RNN: más capas y bidireccionalidad**

Aquí se comparan varias configuraciones:

- **RNN simple**
- **RNN de varias capas**
- **RNN bidireccional**

La comparación ayuda a discutir una pregunta típica:  **¿hasta dónde alcanzan las RNN antes de que convenga pasar a mecanismos de atención?**

In [ ]:
# Extendemos el modelo a una RNN de 3 capas para evaluar mayor capacidad.
rnn_3layer = nn.Sequential(
  EmbeddingPackable(nn.Embedding(vocab_size, D)), #(B, T) -> (B, T, D)
  nn.RNN(D, hidden_nodes, num_layers=3, batch_first=True),  # (B, T, D) -> ((B, T, D), (S, B, D))
  LastTimeStep(rnn_layers=3),  # Reducimos la salida de la RNN a un vector por ejemplo, (B, D)
  nn.Linear(hidden_nodes, classes),  # (B, D) -> (B, classes)
)

rnn_3layer.to(device)
rnn_3layer_results = train_simple_network(rnn_3layer, loss_func, train_packed_loader, test_loader=test_packed_loader, score_funcs={'Accuracy': accuracy_score}, device=device, epochs=20, lr=0.01, disable_tqdm=True)

In [ ]:
# Comparamos la RNN de una capa con la RNN de tres capas.
sns.lineplot(x='epoch', y='test Accuracy', data=packed_train, label='RNN: 1-Layer')
sns.lineplot(x='epoch', y='test Accuracy', data=rnn_3layer_results, label='RNN: 3-Layer')

In [ ]:
# Probamos una RNN profunda y bidireccional para capturar contexto desde ambos sentidos.
rnn_3layer_bidir = nn.Sequential(
  EmbeddingPackable(nn.Embedding(vocab_size, D)), #(B, T) -> (B, T, D)
  nn.RNN(D, hidden_nodes, num_layers=3, batch_first=True, bidirectional=True),  # (B, T, D) -> ((B, T, D), (S, B, D))
  LastTimeStep(rnn_layers=3, bidirectional=True),  # Reducimos la salida bidireccional a un vector por ejemplo, (B, 2D)
  nn.Linear(hidden_nodes*2, classes),  # (B, 2D) -> (B, classes)
)

rnn_3layer_bidir.to(device)
rnn_3layer_bidir_results = train_simple_network(rnn_3layer_bidir, loss_func, train_packed_loader, test_loader=test_packed_loader, score_funcs={'Accuracy': accuracy_score}, device=device, epochs=20, lr=0.01, disable_tqdm=True)

In [ ]:
# Comparamos las tres variantes: 1 capa, 3 capas y 3 capas bidireccional.
sns.lineplot(x='epoch', y='test Accuracy', data=packed_train, label='RNN: 1-Layer')
sns.lineplot(x='epoch', y='test Accuracy', data=rnn_3layer_results, label='RNN: 3-Layer')
sns.lineplot(x='epoch', y='test Accuracy', data=rnn_3layer_bidir_results, label='RNN: 3-Layer BiDir')

#### **6. De RNN a los mecanismos de atención**

Una RNN resume la historia de la secuencia en estados ocultos. Eso funciona, pero en secuencias largas aparecen problemas:

- la información lejana puede degradarse,
- el procesamiento es secuencial,
- y es difícil que cada posición consulte directamente cualquier otra posición.

La **atención** cambia esa lógica: cada token puede comparar su representación con las demás y decidir **a cuáles mirar más**.

El siguiente bloque agrega un ejemplo corto de **self-attention** sobre una secuencia de caracteres.  
No es todavía un transformer completo, pero sí muestra la idea central de lo que viene: **queries, keys, values y matriz de atención**.

In [ ]:
# Ejemplo mínimo de self-attention sobre una secuencia.
# La idea es mostrar cómo cada posición puede "mirar" a las demás sin depender
# únicamente del último estado oculto de una RNN.

embedding_dim = 32
nombre_ejemplo = "martinez"

# Convertimos el nombre en una secuencia de índices y agregamos dimensión batch.
input_ids = dataset.string2InputVec(nombre_ejemplo).reshape(1, -1).to(device)

# Definimos capas simples para producir queries, keys y values.
embedding_layer = nn.Embedding(vocab_size, embedding_dim).to(device)
W_q = nn.Linear(embedding_dim, embedding_dim, bias=False).to(device)
W_k = nn.Linear(embedding_dim, embedding_dim, bias=False).to(device)
W_v = nn.Linear(embedding_dim, embedding_dim, bias=False).to(device)

with torch.no_grad():
    # 1) Embedding de la secuencia: (B, T) -> (B, T, D)
    X = embedding_layer(input_ids)

    # 2) Proyecciones lineales para obtener queries, keys y values.
    Q = W_q(X)
    K = W_k(X)
    V = W_v(X)

    # 3) Producto punto escalado: cada posición compara su consulta con todas las claves.
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (embedding_dim ** 0.5)

    # 4) Softmax para convertir puntajes en pesos de atención.
    attn = torch.softmax(scores, dim=-1)

    # 5) Combinación ponderada de los values.
    context = torch.matmul(attn, V)

print("Forma de X (embeddings):", X.shape)
print("Forma de la matriz de atención:", attn.shape)
print("Forma del contexto:", context.shape)

# Visualizamos la matriz de atención para interpretar qué posiciones atienden a cuáles.
df_attn = pd.DataFrame(
    attn[0].cpu().numpy(),
    index=[f"{i}:{c}" for i, c in enumerate(nombre_ejemplo)],
    columns=[f"{i}:{c}" for i, c in enumerate(nombre_ejemplo)],
)

plt.figure(figsize=(7, 5))
sns.heatmap(df_attn, annot=True, fmt=".2f", cmap="Blues")
plt.title("Ejemplo de self-attention sobre una secuencia de caracteres")
plt.xlabel("Posiciones atendidas (keys)")
plt.ylabel("Posiciones que consultan (queries)")
plt.tight_layout()
plt.show()

#### **Cómo leer este ejemplo**

- En una **RNN**, la información suele fluir a través del estado oculto.
- En **self-attention**, cada posición construye una consulta (**query**) y la compara con todas las claves (**keys**).
- El resultado es una **matriz de atención**, que indica qué tanto pesa cada token al construir una nueva representación.
- Esa nueva representación se obtiene al combinar los **values** con esos pesos.

Este es el paso conceptual que abre la puerta a conceptos que aparecen más adelante:

- **self-attention**
- **multi-head attention**
- **residual connections**
- **layer normalization**
- **bloque transformer**

### **6. Ejercicios**

1. **Pesos compartidos vs red normal**
   Entrena nuevamente `model_regular` y `model_shared` con MNIST y compara:

   * `test Accuracy`
   * tiempo de entrenamiento
   * cantidad de parámetros entrenables
     Luego explica qué ventaja aporta la **compartición de pesos** y en qué casos podría limitar la capacidad del modelo.

2. **Inspección del dataset de nombres por idioma**
   Usando `LanguageNameDataset`, elige 10 ejemplos del conjunto y muestra:

   * nombre original,
   * secuencia de índices generada,
   * etiqueta del idioma.
     Después responde qué papel cumplen `alphabet`, `string2InputVec` y la limpieza de caracteres Unicode.

3. **Análisis del embedding**
   * Cambia el tamaño del embedding (`D`) por al menos tres valores distintos, por ejemplo `16`, `64` y `128`.
   * Entrena la primera RNN y compara el rendimiento.
   * Concluye cómo afecta la dimensión del embedding a la representación de los caracteres.

4. **Primera RNN: interpretación de predicciones**
   Usa `first_rnn` para predecir el idioma de 8 nombres nuevos, algunos fáciles y otros ambiguos.
   Para cada nombre:

   * muestra las probabilidades por clase,
   * indica la predicción final,
   * comenta si el resultado parece razonable.
   * Finalmente, identifica dos casos donde el modelo se confunda.

5. **Padding y packed sequences**
   * Explica paso a paso qué hace la función `pad_and_pack`.
   Luego modifica el experimento para entrenar:
   * una RNN con `batch_size = 1`,
   * una RNN con `PackedSequence`.
   * Compara precisión y tiempo total, y justifica por qué `packed sequences` mejora la eficiencia.

6. **Rol de `LastTimeStep`**
   * Reemplaza temporalmente `LastTimeStep` por otra estrategia simple, como usar la salida de todos los pasos y aplicar un promedio temporal.
   * Entrena el modelo y compara con la versión original.
   * Luego responde por qué usar el **último estado oculto** puede ser suficiente en algunos problemas de clasificación secuencial.

7. **Profundidad y bidireccionalidad en RNN**
   Compara estos tres modelos del cuaderno:

   * `rnn_packed`
   * `rnn_3layer`
   * `rnn_3layer_bidir`

 Elabora una conclusión sobre:
   * cuál aprende más rápido,
   * cuál logra mejor accuracy,
   * qué aporta una RNN **bidireccional** frente a una unidireccional.

8. **Puente entre RNN y atención**
   A partir del ejemplo final de `self-attention`, explica con tus palabras:

   * cómo una RNN procesa la secuencia,
   * cómo self-attention relaciona cada posición con las demás,
   * por qué la atención puede manejar mejor dependencias largas.

Como parte del ejercicio, representa con un esquema simple el flujo de información en ambos casos.



In [ ]:
## Tus respuestas